# Which view supports each grouping?

**The same cells can be grouped in different ways.** Compare a simulated reference annotation with expression-derived and QC-derived clusters. All three have five groups; the reference is a teaching label, not biological ground truth.

New to ITCA? [Start with the concept](00_concepts.ipynb).

In [ ]:
from IPython.display import display
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import gauge_sc as gauge

adata = gauge.datasets.audit_example()
group = adata.obs["group"].astype(str)
adata.obs["reference"] = group.where(~group.str.startswith("Q"), "Q")
adata.obs["expression_clusters"] = KMeans(
    n_clusters=5, n_init=10, random_state=0
).fit_predict(adata.obsm["X_pca"]).astype(str)
qc_columns = ["total_counts", "n_genes_by_counts", "pct_counts_mt"]
qc = adata.obs[qc_columns].to_numpy().copy()
qc[:, :2] = np.log1p(qc[:, :2])
adata.obs["qc_clusters"] = KMeans(
    n_clusters=5, n_init=10, random_state=0
).fit_predict(StandardScaler().fit_transform(qc)).astype(str)

## Evaluate fixed candidates

Candidate generation above is an external step. Evaluation predicts each fixed assignment from each view; it does not rerun clustering inside folds.

In [ ]:
result = gauge.evaluate(
    adata,
    groupby=["reference", "expression_clusters", "qc_clusters"],
    views={
        "molecular": {"obsm": "X_pca"},
        "qc": {"obs": qc_columns, "log1p": qc_columns[:2]},
    },
)
display(result.table.round(3))


The expression-derived grouping and reference should receive strong molecular support; QC-derived clusters should receive stronger QC support. These are **ITCA scores**, not ITCA+ or proof of correct cell types. Group sizes still affect information weights even when group counts match.

For your data, supply existing annotation/clustering columns in one AnnData. To select across different granularities, use [ITCA+ selection](00_select_grouping.ipynb). To inspect individual groups, use [view-specific audit](01_view_specific_audit.ipynb). [Evaluation assumptions](../docs/recipes.md#evaluation-assumptions).

## Work with your agent

[Install the skill and choose an interface](../docs/agents.md). Then try:

> Use $gauge to compare the supplied groupings of these same observations. Choose views that answer my question, show their ITCA profiles, and explain why the preferred grouping can depend on the view. Ask me which distinctions matter if the comparison goal is unclear.

These tutorial settings are an example; adapt them to your question and data.